# Bases de datos NoSQL y MongoDB

**Seminario de actualización · Unidad 6: Bases de datos No Relacionales (NoSQL) y Bases de Datos para IA**

En esta clase recorremos por qué aparecen las bases NoSQL, cuáles son sus modelos de datos, cómo manejan la consistencia en sistemas distribuidos (ACID, BASE, teorema CAP, quórum) y practicamos las operaciones CRUD, la agregación, los índices y el modelado en **MongoDB** desde Python.

### Docker
Para instalar mongodb en Docker se pueden utilizar estos dos comandos:
```
docker pull mongodb/mongodb-community-server:latest 
docker run --name mongodb -p 27017:27017 --volume datamongo:/data/db -d mongodb/mongodb-community-server:latest
```

**Cómo usar este notebook**

1. Ejecutar las celdas en orden, de arriba hacia abajo (`Shift + Enter`).
2. La celda de configuración intenta conectarse a un servidor MongoDB local (`mongodb://localhost:27017`). Si no lo encuentra, usa **mongomock**, una simulación en memoria que acepta la misma sintaxis de `pymongo`. Todo el notebook funciona en ambos modos, salvo `explain()`, que solo existe en un servidor real.
3. Cada operación se muestra primero como se escribe en la consola **mongosh** (JavaScript) y luego como se ejecuta desde Python con **pymongo**. La lógica es la misma; cambian detalles de sintaxis.

| mongosh (JavaScript) | pymongo (Python) |
|---|---|
| `db.alumnos.insertOne({...})` | `db.alumnos.insert_one({...})` |
| `{ legajo: 1024 }` | `{"legajo": 1024}` (las claves van entre comillas) |
| `true`, `false`, `null` | `True`, `False`, `None` |
| `.sort({ apellido: 1 })` | `.sort("apellido", 1)` o `.sort([("apellido", 1)])` |
| `ISODate("2025-03-10")` | `datetime(2025, 3, 10)` |

## 0. Preparación del entorno

La siguiente celda instala solo los paquetes que falten. Conviene ejecutarla dentro del entorno virtual del seminario (ver el Anexo del apunte).

In [1]:
import importlib.util, subprocess, sys

PAQUETES = {"pymongo": "pymongo", "mongomock": "mongomock", "pandas": "pandas"}
faltan = [pip for modulo, pip in PAQUETES.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
print("Dependencias listas")

Instalando: pymongo, mongomock
Dependencias listas


In [2]:
# Configuración de la conexión: cambiar MONGO_URI si el servidor está en otro equipo o puerto.
MONGO_URI = "mongodb://localhost:27017"
BASE = "instituto"

from datetime import datetime
from pprint import pprint
import pandas as pd
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError, DuplicateKeyError

try:
    cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=2000)
    cliente.admin.command("ping")
    MODO = "MongoDB real"
except Exception:
    import mongomock
    cliente = mongomock.MongoClient()
    MODO = "simulado (mongomock)"

db = cliente[BASE]
print(f"Conectado en modo: {MODO} · base de datos: {BASE}")


def ver(cursor_o_lista):
    """Muestra el resultado de una consulta como tabla de pandas."""
    return pd.DataFrame(list(cursor_o_lista))

Conectado en modo: MongoDB real · base de datos: instituto


## 1. Del modelo relacional a NoSQL

### Big Data: las cinco V

Meier y Kaufmann definen Big Data como volúmenes de datos que superan los límites del software convencional. Se lo caracteriza con cinco dimensiones:

| Dimensión | Qué describe | Ejemplo |
|---|---|---|
| Volumen | De terabytes a zettabytes. | Registros de navegación de un sitio de comercio electrónico. |
| Variedad | Datos estructurados, semiestructurados y no estructurados. | Publicaciones en redes sociales con fotos, etiquetas y comentarios. |
| Velocidad | Flujos que deben procesarse en tiempo real. | Lecturas de sensores de tránsito o medidores inteligentes. |
| Valor | El dato como activo de la organización. | Recomendaciones de productos que aumentan las ventas. |
| Veracidad | Más datos no significa mejores análisis. | Reseñas falsas o duplicadas en un sitio de opiniones. |

### Las dos tensiones del modelo relacional

Sadalage y Fowler (*NoSQL Distilled*) identifican dos tensiones:

- **Desajuste de impedancia:** un pedido, con sus líneas, el cliente y la dirección de envío, es una sola unidad para el programa, pero se reparte en varias tablas normalizadas y se vuelve a armar con `JOIN` en cada lectura.
- **Escalado:** un servidor relacional escala bien hacia arriba (*scale up*, un equipo más potente) y mal hacia los costados (*scale out*, un clúster de equipos económicos).

### Ejemplo: el mismo pedido en tablas y como agregado

Un **agregado** es un conjunto de datos relacionados que la aplicación trata como una unidad: se lee entero, se modifica entero y se guarda junto. La celda siguiente arma el mismo pedido de las dos maneras.

In [3]:
# Modelo relacional: tres tablas normalizadas
clientes = pd.DataFrame([{"ClienteId": 1, "Nombre": "Ana Pérez", "Localidad": "CABA"}])
pedidos = pd.DataFrame([{"PedidoId": 501, "ClienteId": 1, "Fecha": "2026-10-15", "Envio": "Av. Rivadavia 1234"}])
lineas = pd.DataFrame([
    {"PedidoId": 501, "Producto": "Teclado", "Cantidad": 1, "Precio": 25000},
    {"PedidoId": 501, "Producto": "Mouse", "Cantidad": 2, "Precio": 9000},
])

# Para leer "el pedido" hay que reunir las tres tablas (el equivalente a dos JOIN)
pedido_relacional = pedidos.merge(clientes, on="ClienteId").merge(lineas, on="PedidoId")
display(pedido_relacional)

# Modelo orientado a agregados: un solo documento con todo lo que se lee junto
pedido_documento = {
    "_id": 501,
    "fecha": "2026-10-15",
    "cliente": {"id": 1, "nombre": "Ana Pérez", "localidad": "CABA"},
    "envio": "Av. Rivadavia 1234",
    "lineas": [
        {"producto": "Teclado", "cantidad": 1, "precio": 25000},
        {"producto": "Mouse", "cantidad": 2, "precio": 9000},
    ],
}
pprint(pedido_documento, sort_dicts=False)

,PedidoId,ClienteId,Fecha,Envio,Nombre,Localidad,Producto,Cantidad,Precio
0,501,1,2026-10-15,Av. Rivadavia 1234,Ana Pérez,CABA,Teclado,1,25000
1,501,1,2026-10-15,Av. Rivadavia 1234,Ana Pérez,CABA,Mouse,2,9000


{'_id': 501,
 'fecha': '2026-10-15',
 'cliente': {'id': 1, 'nombre': 'Ana Pérez', 'localidad': 'CABA'},
 'envio': 'Av. Rivadavia 1234',
 'lineas': [{'producto': 'Teclado', 'cantidad': 1, 'precio': 25000},
            {'producto': 'Mouse', 'cantidad': 2, 'precio': 9000}]}


Si cada agregado se guarda completo en un mismo nodo del clúster, la base puede repartir los pedidos entre muchos servidores sin necesidad de reunirlos con `JOIN`.

## 2. NoSQL: definición y modelos

> **Definición clave.** Meier y Kaufmann llaman NoSQL a todo enfoque no relacional que cumple dos criterios: **los datos no se almacenan en tablas** y **el lenguaje de consulta no es SQL**. También se lee como *Not only SQL*: estas tecnologías conviven con las bases relacionales, no las reemplazan.

Características habituales: modelos clave-valor, documentos, columnas o grafos; arquitectura distribuida con réplicas; procesamiento en paralelo (MapReduce); consistencia configurable; esquema flexible.

| Modelo | Unidad de dato | Casos de uso | Ejemplos |
|---|---|---|---|
| Clave-valor | Un valor opaco asociado a una clave. Solo se accede por clave. | Sesiones de usuario, caché, carritos. | Redis, DynamoDB |
| Documental | Un documento JSON/BSON con estructura anidada. Se consulta por cualquier campo. | Catálogos, perfiles, contenidos. | MongoDB, CouchDB |
| Familia de columnas | Filas con columnas agrupadas en familias. | Series de tiempo, eventos, IoT. | Cassandra, HBase |
| Grafos | Nodos y aristas con propiedades. | Redes sociales, recomendaciones, fraude. | Neo4j |

Tres de los cuatro modelos están orientados a agregados; el de grafos pone el foco en las relaciones.

## 3. Consistencia en sistemas distribuidos

### ACID y BASE

| Aspecto | ACID | BASE |
|---|---|---|
| Prioridad | Consistencia | Disponibilidad |
| Transacciones | Estrictas, con bloqueos o control optimista | Relajadas, con versiones múltiples |
| Escalado | Principalmente vertical | Horizontal |
| Lectura después de escribir | Siempre ve el último valor | Puede ver un valor anterior por un tiempo |
| Uso típico | Finanzas, inventario, facturación | Redes sociales, catálogos, registros |

**BASE** = *Basically Available, Soft state, Eventually consistent*: el sistema sigue respondiendo, acepta que las réplicas tengan por un tiempo versiones distintas y garantiza que, sin nuevas escrituras, todas convergen al mismo valor (**consistencia eventual**).

### El teorema CAP

Brewer (2000), demostrado por Gilbert y Lynch (2002): un sistema distribuido masivo solo puede garantizar **dos de tres** propiedades.

- **C**onsistencia: toda lectura obtiene el último valor escrito.
- **A**vailability (disponibilidad): el sistema responde siempre, en un tiempo aceptable.
- Tolerancia a **P**articiones: sigue funcionando aunque fallen nodos o enlaces.

| Combinación | Ejemplo | Justificación |
|---|---|---|
| CA | Bolsa de valores | Datos exactos y respuesta inmediata; bases relacionales ACID. |
| CP | Red nacional de cajeros automáticos | Prefiere demorar una operación antes que mostrar un saldo incorrecto. |
| AP | DNS | Debe estar siempre disponible; un cambio puede tardar en propagarse. |

Como las particiones de red ocurren, la decisión real es qué hacer cuando ocurren: priorizar la consistencia o la disponibilidad.

### Consistencia ajustable: el quórum

- **N**: cantidad de réplicas de cada dato.
- **W**: réplicas que deben confirmar una escritura.
- **R**: réplicas que se leen para responder.

Si **W + R > N**, toda lectura toca al menos una réplica con la última escritura (consistencia por quórum). Si **W + R ≤ N**, la consistencia es eventual.

La siguiente simulación lo comprueba: se escribe una versión nueva en W réplicas elegidas al azar (las otras todavía tienen la versión vieja) y se leen R réplicas al azar. Si ninguna de las leídas tiene la versión nueva, la lectura devolvió un dato viejo.

In [4]:
import random

random.seed(42)


def es_consistente(N, W, R):
    return W + R > N


def prob_lectura_vieja(N, W, R, pruebas=20_000):
    viejas = 0
    for _ in range(pruebas):
        version = [0] * N                          # todas las réplicas con la versión vieja
        for i in random.sample(range(N), W):       # la escritura confirmó en W réplicas
            version[i] = 1
        leidas = random.sample(range(N), R)        # la lectura consulta R réplicas
        if max(version[i] for i in leidas) == 0:   # ninguna tenía la versión nueva
            viejas += 1
    return viejas / pruebas


filas = []
for N in (3, 5):
    for W in range(1, N + 1):
        for R in range(1, N + 1):
            filas.append({"N": N, "W": W, "R": R, "W+R>N": es_consistente(N, W, R),
                          "% lecturas viejas": round(100 * prob_lectura_vieja(N, W, R), 1)})
quorum = pd.DataFrame(filas)
quorum[quorum.N == 3]

,N,W,R,W+R>N,% lecturas viejas
0,3,1,1,False,66.5
1,3,1,2,False,33.6
2,3,1,3,True,0.0
3,3,2,1,False,33.0
4,3,2,2,True,0.0
5,3,2,3,True,0.0
6,3,3,1,True,0.0
7,3,3,2,True,0.0
8,3,3,3,True,0.0


Con N = 3, escribir en 2 y leer de 2 nunca devuelve un valor viejo (2 + 2 > 3). Escribir en 1 y leer de 1 es más rápido, pero en dos de cada tres lecturas devuelve el valor anterior.

**Ejercicio (apunte):** con N = 5, indicar tres combinaciones de W y R que garanticen consistencia y una que no lo haga. La tabla siguiente sirve para verificar la respuesta.

In [5]:
quorum[quorum.N == 5].pivot(index="W", columns="R", values="% lecturas viejas")

R,1,2,3,4,5
W,,,,,
1,80.5,60.4,39.6,20.0,0.0
2,60.2,30.2,10.0,0.0,0.0
3,40.4,9.7,0.0,0.0,0.0
4,20.1,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0


> **Regla general:** NoSQL no es "mejor" ni "peor" que SQL. Cada modelo resuelve un problema distinto, y una misma aplicación puede combinar varios motores (**persistencia políglota**).

## 4. MongoDB: documentos, colecciones y BSON

| SQL Server | MongoDB | Observación |
|---|---|---|
| Base de datos | Base de datos | Contenedor lógico. |
| Tabla | Colección | No exige que todos los documentos tengan los mismos campos. |
| Fila | Documento | Puede contener subdocumentos y arrays. |
| Columna | Campo | Se define al insertar, no antes. |
| Clave primaria | Campo `_id` | Si no se indica, MongoDB genera un `ObjectId`. |
| JOIN | Documento embebido o `$lookup` | Se prefiere embeber lo que se lee junto. |
| GROUP BY | Framework de agregación (`$group`) | Ver la sección 6. |

Un documento se escribe en JSON y se almacena en **BSON** (JSON binario), que agrega fechas, enteros de 32 y 64 bits, `Decimal128` y `ObjectId`. El tamaño máximo de un documento es de 16 MB.

En mongosh, el documento del apunte se ve así:

```javascript
{
  _id: ObjectId("6710a1f2c3b4d5e6f7a8b9c0"),
  legajo: 1024, nombre: "Ana", apellido: "Pérez",
  carrera: "Análisis de Sistemas",
  materias: [ { codigo: "SEM", nombre: "Seminario", nota: 8 },
              { codigo: "BD2", nombre: "Bases de Datos II", nota: 9 } ],
  domicilio: { calle: "Av. Rivadavia 1234", localidad: "CABA" },
  fechaAlta: ISODate("2025-03-10")
}
```

Para empezar desde cero en cada ejecución, la celda siguiente borra la colección `alumnos` si existía (equivale a `db.alumnos.drop()`).

In [6]:
db.alumnos.drop()
db.cursos.drop()
print("Colecciones actuales:", db.list_collection_names())   # en mongosh: show collections

Colecciones actuales: []


## 5. Operaciones CRUD

### 5.1 Inserción: `insertOne` e `insertMany`

```javascript
db.coleccion.insertOne(documento, { writeConcern })
db.coleccion.insertMany([documento1, documento2, ...], { ordered: true })
```

Si la colección no existe, MongoDB la crea con la primera inserción.

In [7]:
ana = {
    "legajo": 1024, "nombre": "Ana", "apellido": "Pérez",
    "carrera": "Análisis de Sistemas", "edad": 21,
    "materias": [
        {"codigo": "SEM", "nombre": "Seminario", "nota": 8},
        {"codigo": "BD2", "nombre": "Bases de Datos II", "nota": 9},
    ],
    "domicilio": {"calle": "Av. Rivadavia 1234", "localidad": "CABA"},
    "fechaAlta": datetime(2025, 3, 10),
}
resultado = db.alumnos.insert_one(ana)
print("_id generado:", resultado.inserted_id, "· tipo:", type(resultado.inserted_id).__name__)

_id generado: 6acfc6d5ba88f6a43834c7b5 · tipo: ObjectId


Ahora cargamos más alumnos con `insert_many`. Algunos tienen el array `materias` y otros no; algunos tienen `becada`, otros no: el **esquema es flexible** (ejercicio 1 del apunte).

In [8]:
otros = [
    {"legajo": 1025, "nombre": "Luis", "apellido": "Gómez", "carrera": "Análisis de Sistemas", "edad": 22,
     "materias": [{"codigo": "SEM", "nombre": "Seminario", "nota": 6}, {"codigo": "PRG", "nombre": "Programación", "nota": 7}],
     "domicilio": {"calle": "Mitre 450", "localidad": "Avellaneda"}},
    {"legajo": 1026, "nombre": "Sofía", "apellido": "Ruiz", "carrera": "Análisis de Sistemas", "edad": 25, "becada": True,
     "materias": [{"codigo": "SEM", "nombre": "Seminario", "nota": 9}, {"codigo": "BD2", "nombre": "Bases de Datos II", "nota": 10}],
     "domicilio": {"calle": "Corrientes 3200", "localidad": "CABA"}},
    {"legajo": 1027, "nombre": "Martín", "apellido": "Acosta", "carrera": "Redes", "edad": 19,
     "materias": [{"codigo": "RED", "nombre": "Redes I", "nota": 5}]},
    {"legajo": 1028, "nombre": "Julieta", "apellido": "Benítez", "carrera": "Redes", "edad": 27,
     "domicilio": {"calle": "Belgrano 88", "localidad": "Lanús"}},
    {"legajo": 1029, "nombre": "Tomás", "apellido": "Castro", "carrera": "Análisis de Sistemas", "edad": 17},
    {"legajo": 1030, "nombre": "Valentina", "apellido": "Díaz", "carrera": "Análisis de Sistemas", "edad": 23,
     "materias": [{"codigo": "BD2", "nombre": "Bases de Datos II", "nota": 3}, {"codigo": "PRG", "nombre": "Programación", "nota": 8}]},
    {"legajo": 1031, "nombre": "Diego", "apellido": "Ferreyra", "carrera": "Redes", "edad": 30, "becada": False,
     "materias": [{"codigo": "RED", "nombre": "Redes I", "nota": 9}, {"codigo": "SEM", "nombre": "Seminario", "nota": 7}]},
    {"legajo": 1032, "nombre": "Camila", "apellido": "Pereyra", "carrera": "Análisis de Sistemas", "edad": 20,
     "materias": [{"codigo": "SEM", "nombre": "Seminario", "nota": 9}],
     "domicilio": {"calle": "San Martín 1500", "localidad": "CABA"}},
    {"legajo": 1033, "nombre": "Nicolás", "apellido": "Morales", "carrera": "Análisis de Sistemas", "edad": 24,
     "materias": [{"codigo": "PRG", "nombre": "Programación", "nota": 4}, {"codigo": "BD2", "nombre": "Bases de Datos II", "nota": 6}]},
    {"legajo": 1034, "nombre": "Lucía", "apellido": "Paz", "carrera": "Redes", "edad": 16},
]
resultado = db.alumnos.insert_many(otros)
print("Insertados:", len(resultado.inserted_ids), "· total en la colección:", db.alumnos.count_documents({}))

Insertados: 10 · total en la colección: 11


### 5.2 Consulta: `find`

```javascript
db.coleccion.find(filtro, proyeccion).sort({ campo: 1 | -1 }).skip(n).limit(n)
```

- El **filtro** cumple el papel del `WHERE`.
- La **proyección** cumple el papel de la lista de columnas del `SELECT` (1 = mostrar, 0 = ocultar).
- `sort`, `skip` y `limit` cumplen el papel de `ORDER BY` y `OFFSET ... FETCH`.

In [9]:
# db.alumnos.find()   -> todos (mostramos solo algunos campos para que entre en pantalla)
ver(db.alumnos.find({}, {"_id": 0, "legajo": 1, "nombre": 1, "apellido": 1, "carrera": 1, "edad": 1, "becada": 1}))

,legajo,nombre,apellido,carrera,edad,becada
0,1024,Ana,Pérez,Análisis de Sistemas,21,NaN
1,1025,Luis,Gómez,Análisis de Sistemas,22,NaN
2,1026,Sofía,Ruiz,Análisis de Sistemas,25,True
3,1027,Martín,Acosta,Redes,19,NaN
4,1028,Julieta,Benítez,Redes,27,NaN
5,1029,Tomás,Castro,Análisis de Sistemas,17,NaN
6,1030,Valentina,Díaz,Análisis de Sistemas,23,NaN
7,1031,Diego,Ferreyra,Redes,30,False
8,1032,Camila,Pereyra,Análisis de Sistemas,20,NaN
9,1033,Nicolás,Morales,Análisis de Sistemas,24,NaN


In [10]:
# Igualdad: db.alumnos.find({ carrera: "Redes" })
ver(db.alumnos.find({"carrera": "Redes"}, {"_id": 0, "legajo": 1, "nombre": 1, "edad": 1}))

,legajo,nombre,edad
0,1027,Martín,19
1,1028,Julieta,27
2,1031,Diego,30
3,1034,Lucía,16


In [11]:
# Rango: db.alumnos.find({ edad: { $gte: 21, $lt: 30 } })
ver(db.alumnos.find({"edad": {"$gte": 21, "$lt": 30}}, {"_id": 0, "nombre": 1, "edad": 1}))

,nombre,edad
0,Ana,21
1,Luis,22
2,Sofía,25
3,Julieta,27
4,Valentina,23
5,Nicolás,24


In [12]:
# OR: db.alumnos.find({ $or: [ { becada: true }, { edad: { $gt: 24 } } ] })
ver(db.alumnos.find({"$or": [{"becada": True}, {"edad": {"$gt": 24}}]}, {"_id": 0, "nombre": 1, "edad": 1, "becada": 1}))

,nombre,edad,becada
0,Sofía,25,True
1,Julieta,27,NaN
2,Diego,30,False


In [13]:
# Campo de un subdocumento (notación de punto): db.alumnos.find({ "domicilio.localidad": "CABA" })
ver(db.alumnos.find({"domicilio.localidad": "CABA"}, {"_id": 0, "nombre": 1, "domicilio": 1}))

,nombre,domicilio
0,Ana,"{'calle': 'Av. Rivadavia 1234', 'localidad': '..."
1,Sofía,"{'calle': 'Corrientes 3200', 'localidad': 'CABA'}"
2,Camila,"{'calle': 'San Martín 1500', 'localidad': 'CABA'}"


In [14]:
# Dentro de un array: alumnos con alguna nota >= 9
# db.alumnos.find({ "materias.nota": { $gte: 9 } })
ver(db.alumnos.find({"materias.nota": {"$gte": 9}}, {"_id": 0, "nombre": 1, "materias": 1}))

,nombre,materias
0,Ana,"[{'codigo': 'SEM', 'nombre': 'Seminario', 'not..."
1,Sofía,"[{'codigo': 'SEM', 'nombre': 'Seminario', 'not..."
2,Diego,"[{'codigo': 'RED', 'nombre': 'Redes I', 'nota'..."
3,Camila,"[{'codigo': 'SEM', 'nombre': 'Seminario', 'not..."


In [15]:
# Proyección, orden y paginación:
# db.alumnos.find({}, { nombre: 1, legajo: 1, _id: 0 }).sort({ apellido: 1 }).skip(2).limit(5)
ver(db.alumnos.find({}, {"_id": 0, "apellido": 1, "nombre": 1, "legajo": 1})
              .sort("apellido", ASCENDING).skip(2).limit(5))

,legajo,nombre,apellido
0,1029,Tomás,Castro
1,1030,Valentina,Díaz
2,1031,Diego,Ferreyra
3,1025,Luis,Gómez
4,1033,Nicolás,Morales


Los operadores de consulta comienzan con `$` y tienen equivalente en SQL (Sarasa, cap. IV):

| Operador MongoDB | Equivalente SQL | Ejemplo |
|---|---|---|
| `$eq`, `$ne` | `=`, `<>` | `{ carrera: { $ne: "Contador" } }` |
| `$gt`, `$gte`, `$lt`, `$lte` | `>`, `>=`, `<`, `<=` | `{ edad: { $gte: 18 } }` |
| `$in`, `$nin` | `IN`, `NOT IN` | `{ codigo: { $in: ["SEM", "BD2"] } }` |
| `$and`, `$or`, `$not` | `AND`, `OR`, `NOT` | `{ $or: [ {...}, {...} ] }` |
| `$exists` | `IS NOT NULL` (aproximado) | `{ becada: { $exists: true } }` |
| `$regex` | `LIKE` | `{ apellido: { $regex: "^P" } }` |

In [16]:
# $in, $exists y $regex
print("Cursan SEM o BD2:")
display(ver(db.alumnos.find({"materias.codigo": {"$in": ["SEM", "BD2"]}}, {"_id": 0, "nombre": 1})))
print("Tienen el campo becada (con cualquier valor):")
display(ver(db.alumnos.find({"becada": {"$exists": True}}, {"_id": 0, "nombre": 1, "becada": 1})))
print("Apellido que empieza con P (LIKE 'P%'):")
display(ver(db.alumnos.find({"apellido": {"$regex": "^P"}}, {"_id": 0, "nombre": 1, "apellido": 1})))

Cursan SEM o BD2:


,nombre
0,Ana
1,Luis
2,Sofía
3,Valentina
4,Diego
5,Camila
6,Nicolás


Tienen el campo becada (con cualquier valor):


,nombre,becada
0,Sofía,True
1,Diego,False


Apellido que empieza con P (LIKE 'P%'):


,nombre,apellido
0,Ana,Pérez
1,Camila,Pereyra
2,Lucía,Paz


**Para pensar:** ¿cómo se escribe en MongoDB `SELECT nombre FROM alumnos WHERE edad > 21 ORDER BY nombre`? Intentarlo antes de ejecutar la solución.

In [17]:
# Solución
# mongosh: db.alumnos.find({ edad: { $gt: 21 } }, { nombre: 1, _id: 0 }).sort({ nombre: 1 })
ver(db.alumnos.find({"edad": {"$gt": 21}}, {"_id": 0, "nombre": 1}).sort("nombre", ASCENDING))

,nombre
0,Diego
1,Julieta
2,Luis
3,Nicolás
4,Sofía
5,Valentina


### 5.3 Actualización: `updateOne`, `updateMany`, `replaceOne`

```javascript
db.coleccion.updateOne(filtro, actualizacion, { upsert: false })
db.coleccion.updateMany(filtro, actualizacion, { upsert: false })
db.coleccion.replaceOne(filtro, documentoNuevo)
```

La actualización se expresa con **modificadores**, que indican qué cambiar sin reemplazar el documento completo:

| Modificador | Efecto | Equivalente aproximado en SQL |
|---|---|---|
| `$set` | Asigna un valor; crea el campo si no existe. | `SET campo = valor` |
| `$unset` | Elimina un campo. | `SET campo = NULL` |
| `$inc` | Suma un valor numérico. | `SET campo = campo + n` |
| `$push` | Agrega un elemento a un array. | `INSERT` en la tabla hija |
| `$pull` | Quita elementos de un array que cumplen una condición. | `DELETE` en la tabla hija |
| `$addToSet` | Agrega al array solo si no existe. | `INSERT ... WHERE NOT EXISTS` |

In [18]:
# $set: db.alumnos.updateOne({ legajo: 1025 }, { $set: { edad: 23 } })
r = db.alumnos.update_one({"legajo": 1025}, {"$set": {"edad": 23}})
print("Coincidieron:", r.matched_count, "· modificados:", r.modified_count)

# $inc sobre varios documentos: db.alumnos.updateMany({ carrera: "Análisis de Sistemas" }, { $inc: { creditos: 2 } })
r = db.alumnos.update_many({"carrera": "Análisis de Sistemas"}, {"$inc": {"creditos": 2}})
print("updateMany con $inc -> modificados:", r.modified_count)

ver(db.alumnos.find({"carrera": "Análisis de Sistemas"}, {"_id": 0, "nombre": 1, "edad": 1, "creditos": 1}))

Coincidieron: 1 · modificados: 1
updateMany con $inc -> modificados: 7


,nombre,edad,creditos
0,Ana,21,2
1,Luis,23,2
2,Sofía,25,2
3,Tomás,17,2
4,Valentina,23,2
5,Camila,20,2
6,Nicolás,24,2


In [19]:
# $push: agregar una materia a Ana
# db.alumnos.updateOne({ legajo: 1024 }, { $push: { materias: { codigo: "PRG", nombre: "Programación", nota: 7 } } })
db.alumnos.update_one({"legajo": 1024},
                      {"$push": {"materias": {"codigo": "PRG", "nombre": "Programación", "nota": 7}}})
pprint(db.alumnos.find_one({"legajo": 1024}, {"_id": 0, "nombre": 1, "materias": 1}), sort_dicts=False)

{'nombre': 'Ana',
 'materias': [{'codigo': 'SEM', 'nombre': 'Seminario', 'nota': 8},
              {'codigo': 'BD2', 'nombre': 'Bases de Datos II', 'nota': 9},
              {'codigo': 'PRG', 'nombre': 'Programación', 'nota': 7}]}


**Subir 1 punto todas las notas de una materia** (ejercicio 4). El operador posicional `$` actualiza el primer elemento del array que cumplió el filtro, en cada documento:

```javascript
db.alumnos.updateMany({ "materias.codigo": "SEM" }, { $inc: { "materias.$.nota": 1 } })
```

En un servidor real también se puede usar `arrayFilters` (`"materias.$[m].nota"` con `arrayFilters: [{ "m.codigo": "SEM" }]`), que actualiza todos los elementos que cumplen la condición. mongomock no implementa `arrayFilters`, por eso usamos `$`.

In [20]:
antes = ver(db.alumnos.aggregate([{"$unwind": "$materias"}, {"$match": {"materias.codigo": "SEM"}},
                                  {"$project": {"_id": 0, "nombre": 1, "nota": "$materias.nota"}}]))
db.alumnos.update_many({"materias.codigo": "SEM"}, {"$inc": {"materias.$.nota": 1}})
despues = ver(db.alumnos.aggregate([{"$unwind": "$materias"}, {"$match": {"materias.codigo": "SEM"}},
                                    {"$project": {"_id": 0, "nombre": 1, "nota": "$materias.nota"}}]))
antes.merge(despues, on="nombre", suffixes=(" antes", " después"))

,nombre,nota antes,nota después
0,Ana,8,9
1,Luis,6,7
2,Sofía,9,10
3,Diego,7,8
4,Camila,9,10


In [21]:
# $addToSet no repite; $pull quita; $unset elimina un campo
db.alumnos.update_one({"legajo": 1026}, {"$addToSet": {"etiquetas": "abanderada"}})
db.alumnos.update_one({"legajo": 1026}, {"$addToSet": {"etiquetas": "abanderada"}})   # no se duplica
db.alumnos.update_one({"legajo": 1030}, {"$pull": {"materias": {"nota": {"$lt": 4}}}})  # quita materias desaprobadas
db.alumnos.update_one({"legajo": 1031}, {"$unset": {"becada": ""}})
pprint(list(db.alumnos.find({"legajo": {"$in": [1026, 1030, 1031]}},
                            {"_id": 0, "nombre": 1, "etiquetas": 1, "materias": 1, "becada": 1})), sort_dicts=False)

[{'nombre': 'Sofía',
  'becada': True,
  'materias': [{'codigo': 'SEM', 'nombre': 'Seminario', 'nota': 10},
               {'codigo': 'BD2', 'nombre': 'Bases de Datos II', 'nota': 10}],
  'etiquetas': ['abanderada']},
 {'nombre': 'Valentina',
  'materias': [{'codigo': 'PRG', 'nombre': 'Programación', 'nota': 8}]},
 {'nombre': 'Diego',
  'materias': [{'codigo': 'RED', 'nombre': 'Redes I', 'nota': 9},
               {'codigo': 'SEM', 'nombre': 'Seminario', 'nota': 8}]}]


**upsert** combina actualización e inserción: si ningún documento cumple el filtro, se inserta uno nuevo. Cumple un papel similar a `MERGE` en T-SQL.

In [22]:
# db.alumnos.updateOne({ legajo: 2000 }, { $set: { nombre: "Nuevo" } }, { upsert: true })
r = db.alumnos.update_one({"legajo": 2000}, {"$set": {"nombre": "Nuevo"}}, upsert=True)
print("Coincidieron:", r.matched_count, "· _id insertado:", r.upserted_id)
r = db.alumnos.update_one({"legajo": 2000}, {"$set": {"nombre": "Nuevo"}}, upsert=True)
print("Segunda vez -> coincidieron:", r.matched_count, "· _id insertado:", r.upserted_id)

Coincidieron: 0 · _id insertado: 6acfc86b4f275b1f1ff3555e
Segunda vez -> coincidieron: 1 · _id insertado: None


### 5.4 Borrado: `deleteOne` y `deleteMany`

```javascript
db.alumnos.deleteOne({ legajo: 1026 })
db.alumnos.deleteMany({ edad: { $lt: 18 } })
db.alumnos.deleteMany({})          // vacía la colección (DELETE sin WHERE)
```

Para no perder datos que usamos después, borramos solo el alumno creado con upsert y los menores de 18.

In [23]:
print("deleteOne:", db.alumnos.delete_one({"legajo": 2000}).deleted_count)
print("Menores de 18:", [a["nombre"] for a in db.alumnos.find({"edad": {"$lt": 18}})])
print("deleteMany:", db.alumnos.delete_many({"edad": {"$lt": 18}}).deleted_count)
print("Quedan:", db.alumnos.count_documents({}), "alumnos")

deleteOne: 1
Menores de 18: ['Tomás', 'Lucía']
deleteMany: 2
Quedan: 9 alumnos


## 6. El framework de agregación

La agregación es una **tubería (pipeline) de etapas**: cada etapa recibe los documentos que produjo la anterior. Es muy parecido al encadenamiento de operaciones de pandas de la Unidad 5.

| Etapa | Equivalente SQL | Función |
|---|---|---|
| `$match` | `WHERE` / `HAVING` | Filtra documentos. |
| `$project` | `SELECT` | Elige, renombra o calcula campos. |
| `$group` | `GROUP BY` | Agrupa y calcula `$sum`, `$avg`, `$min`, `$max`. |
| `$sort` | `ORDER BY` | Ordena. |
| `$limit`, `$skip` | `TOP`, `OFFSET-FETCH` | Pagina. |
| `$unwind` | Sin equivalente directo | Genera un documento por cada elemento de un array. |
| `$lookup` | `LEFT OUTER JOIN` | Trae documentos de otra colección. |

Primero veamos qué hace `$unwind` por sí solo:

In [24]:
ver(db.alumnos.aggregate([
    {"$match": {"legajo": 1024}},
    {"$unwind": "$materias"},
    {"$project": {"_id": 0, "nombre": 1, "materia": "$materias.codigo", "nota": "$materias.nota"}},
]))

,nombre,materia,nota
0,Ana,SEM,9
1,Ana,BD2,9
2,Ana,PRG,7


In [25]:
# Promedio de notas aprobadas por materia, de mayor a menor (ejemplo del apunte)
pipeline = [
    {"$unwind": "$materias"},
    {"$match": {"materias.nota": {"$gte": 4}}},
    {"$group": {"_id": "$materias.codigo",
                "promedio": {"$avg": "$materias.nota"},
                "aprobados": {"$sum": 1}}},
    {"$sort": {"promedio": -1}},
]
promedios_mongo = ver(db.alumnos.aggregate(pipeline))
promedios_mongo

,_id,promedio,aprobados
0,SEM,8.800000,5
1,BD2,8.333333,3
2,RED,7.000000,2
3,PRG,6.500000,4


### Comparación con SQL

La consulta equivalente en T-SQL, sobre un modelo relacional con una tabla `MateriasCursadas`, es:

```sql
SELECT CodigoMateria, AVG(Nota * 1.0) AS Promedio, COUNT(*) AS Aprobados
FROM MateriasCursadas
WHERE Nota >= 4
GROUP BY CodigoMateria
ORDER BY Promedio DESC;
```

Para comprobar que dan lo mismo sin depender de SQL Server, la celda siguiente **normaliza** los documentos en una tabla relacional y ejecuta la misma consulta en SQLite (incluido en Python). En SQL Server la sintaxis es idéntica.

In [26]:
import sqlite3

filas = [{"Legajo": a["legajo"], "CodigoMateria": m["codigo"], "Nota": m["nota"]}
         for a in db.alumnos.find() for m in a.get("materias", [])]
con = sqlite3.connect(":memory:")
pd.DataFrame(filas).to_sql("MateriasCursadas", con, index=False)

promedios_sql = pd.read_sql("""
    SELECT CodigoMateria, AVG(Nota * 1.0) AS Promedio, COUNT(*) AS Aprobados
    FROM MateriasCursadas
    WHERE Nota >= 4
    GROUP BY CodigoMateria
    ORDER BY Promedio DESC;
""", con)
promedios_sql

,CodigoMateria,Promedio,Aprobados
0,SEM,8.800000,5
1,BD2,8.333333,3
2,RED,7.000000,2
3,PRG,6.500000,4


Notar el paso de normalización: en el modelo relacional las materias cursadas son una tabla aparte (una fila por alumno y materia). En MongoDB están embebidas en el documento y `$unwind` hace ese "aplanado" dentro de la consulta.

Otro ejemplo: cantidad de alumnos y edad promedio por carrera, con `$project` para renombrar.

In [27]:
ver(db.alumnos.aggregate([
    {"$group": {"_id": "$carrera", "alumnos": {"$sum": 1}, "edad_promedio": {"$avg": "$edad"}}},
    {"$project": {"_id": 0, "carrera": "$_id", "alumnos": 1, "edad_promedio": 1}},
    {"$sort": {"alumnos": -1}},
])).round(1)

,alumnos,edad_promedio,carrera
0,6,22.7,Análisis de Sistemas
1,3,25.3,Redes


## 7. Índices

Sin índices, MongoDB recorre toda la colección (*collection scan*), igual que un *table scan* en SQL Server. `explain("executionStats")` muestra el plan de ejecución.

```javascript
db.alumnos.createIndex({ legajo: 1 }, { unique: true })   // único
db.alumnos.createIndex({ carrera: 1, apellido: 1 })       // compuesto
db.alumnos.find({ legajo: 1024 }).explain("executionStats")
```

In [28]:
def plan(filtro):
    """Muestra el plan de ejecución. Solo funciona con un servidor MongoDB real."""
    if MODO != "MongoDB real":
        print("explain() no está disponible en modo simulado: ejecutar con un servidor MongoDB para ver el plan.")
        return
    info = db.alumnos.find(filtro).explain()
    etapa = info["queryPlanner"]["winningPlan"]
    etapa = etapa.get("queryPlan", etapa)  # MongoDB 7+ anida el plan en "queryPlan"
    while "inputStage" in etapa:          # bajar hasta la etapa de acceso a datos
        etapa = etapa["inputStage"]
    stats = info.get("executionStats", {})
    print(f"Etapa: {etapa['stage']}  (COLLSCAN = recorre todo; IXSCAN = usa un índice)")
    print("Documentos examinados:", stats.get("totalDocsExamined"), "· devueltos:", stats.get("nReturned"))


print("ANTES del índice")
plan({"legajo": 1024})

ANTES del índice
Etapa: COLLSCAN  (COLLSCAN = recorre todo; IXSCAN = usa un índice)
Documentos examinados: 9 · devueltos: 1


In [29]:
db.alumnos.create_index([("legajo", ASCENDING)], unique=True, name="ux_legajo")
db.alumnos.create_index([("carrera", ASCENDING), ("apellido", ASCENDING)], name="ix_carrera_apellido")
pprint(db.alumnos.index_information())

print("\nDESPUÉS del índice")
plan({"legajo": 1024})

{'_id_': {'key': [('_id', 1)], 'v': 2},
 'ix_carrera_apellido': {'key': [('carrera', 1), ('apellido', 1)], 'v': 2},
 'ux_legajo': {'key': [('legajo', 1)], 'unique': True, 'v': 2}}

DESPUÉS del índice
Etapa: EXPRESS_IXSCAN  (COLLSCAN = recorre todo; IXSCAN = usa un índice)
Documentos examinados: 1 · devueltos: 1


El índice único además protege la integridad, como una restricción `UNIQUE` en SQL Server. También sirve para ver la diferencia entre `ordered=True` y `ordered=False` en `insert_many`:

In [30]:
nuevos = [{"legajo": 1040, "nombre": "Bruno"}, {"legajo": 1024, "nombre": "Duplicado"}, {"legajo": 1041, "nombre": "Carla"}]

try:
    db.alumnos.insert_one({"legajo": 1024, "nombre": "Otra Ana"})
except DuplicateKeyError:
    print("insert_one rechazado: el legajo 1024 ya existe")

try:
    db.alumnos.insert_many(nuevos, ordered=False)    # continúa después del error
except BulkWriteError as e:
    print("ordered=False -> insertados:", e.details["nInserted"], "· errores:", len(e.details["writeErrors"]))

print("Legajos 1040 y 1041 presentes:", db.alumnos.count_documents({"legajo": {"$in": [1040, 1041]}}))
_ = db.alumnos.delete_many({"legajo": {"$in": [1040, 1041]}})

insert_one rechazado: el legajo 1024 ya existe
ordered=False -> insertados: 2 · errores: 1
Legajos 1040 y 1041 presentes: 2


## 8. Modelado: embeber o referenciar

En el modelo relacional se diseña a partir de la normalización. En MongoDB se diseña a partir de **cómo la aplicación lee los datos**.

| Criterio | Embeber | Referenciar |
|---|---|---|
| Tipo de relación | Uno a pocos, del tipo "pertenece a". | Uno a muchos grandes o muchos a muchos. |
| Lectura | Una sola consulta trae todo. | Requiere `$lookup` o una segunda consulta. |
| Actualización | Si el dato se repite, hay que actualizarlo en todos. | Se actualiza en un solo lugar. |
| Límite | El documento no puede superar 16 MB. | Sin límite práctico. |
| Ejemplo | Domicilio de un alumno, líneas de un pedido. | Alumnos y cursos, autores y libros. |

En nuestra colección, `domicilio` está **embebido**. Ahora modelamos los cursos como colección aparte y los **referenciamos** por su `_id`; `$lookup` hace el equivalente a un `LEFT JOIN`.

In [31]:
db.cursos.insert_many([
    {"_id": "SEM", "nombre": "Seminario de actualización", "docente": "Insfran", "horas": 2},
    {"_id": "BD2", "nombre": "Bases de Datos II", "docente": "López", "horas": 4},
    {"_id": "PRG", "nombre": "Programación", "docente": "García", "horas": 4},
    {"_id": "RED", "nombre": "Redes I", "docente": "Suárez", "horas": 3},
])

# Inscripciones referenciadas: el alumno guarda solo los códigos de curso
db.alumnos.update_one({"legajo": 1024}, {"$set": {"inscripto_en": ["SEM", "BD2"]}})

ver(db.alumnos.aggregate([
    {"$match": {"legajo": 1024}},
    {"$lookup": {"from": "cursos", "localField": "inscripto_en", "foreignField": "_id", "as": "cursos"}},
    {"$unwind": "$cursos"},
    {"$project": {"_id": 0, "alumno": "$nombre", "curso": "$cursos.nombre", "docente": "$cursos.docente"}},
]))

,alumno,curso,docente
0,Ana,Bases de Datos II,López
1,Ana,Seminario de actualización,Insfran


Si el docente de un curso cambia, se actualiza **un** documento en `cursos`. Si estuviera embebido en cada alumno, habría que actualizar todos.

> **Regla general:** en una base documental se modela para la consulta. Si dos datos se leen siempre juntos, conviene embeberlos; si se actualizan por separado o crecen sin límite, conviene referenciarlos.

## 9. Replicación y sharding

- **Replicación (replica set):** un nodo primario recibe las escrituras y los secundarios copian sus operaciones. Si el primario falla, los secundarios eligen uno nuevo (Sarasa, cap. VII).
- **Fragmentación (sharding):** una colección se reparte entre varios servidores según una **clave de sharding**. Una buena clave distribuye la carga en forma pareja; una mala concentra las escrituras en un único servidor (Sarasa, cap. VIII).

La simulación siguiente reparte 3.000 alumnos nuevos (legajos consecutivos) entre 3 shards con dos estrategias:

- **Por rangos de legajo:** cada shard tiene un rango fijo definido con los datos que ya existían (hasta 3.333, hasta 6.666, el resto).
- **Por hash del legajo:** el shard se elige con el resto de dividir un hash por 3.

In [32]:
import hashlib


def shard_por_rango(legajo):
    return 0 if legajo < 3333 else 1 if legajo < 6666 else 2


def shard_por_hash(legajo):
    return int(hashlib.md5(str(legajo).encode()).hexdigest(), 16) % 3


nuevos_legajos = range(10_000, 13_000)     # los legajos nuevos siempre son mayores que los existentes
pd.DataFrame({
    "por rango": pd.Series([shard_por_rango(l) for l in nuevos_legajos]).value_counts().sort_index(),
    "por hash": pd.Series([shard_por_hash(l) for l in nuevos_legajos]).value_counts().sort_index(),
}).fillna(0).astype(int).rename_axis("shard")

,por rango,por hash
shard,,
0,0,1037
1,0,1019
2,3000,944


Con una clave **creciente** y reparto por rangos, todas las escrituras nuevas caen en el último shard (un "punto caliente"). El hash las reparte de forma pareja, a cambio de que las consultas por rango de legajo deban consultar todos los shards.

## 10. MongoDB desde Python con pandas

Igual que con SQL Server en la Unidad 5, el resultado de una consulta puede llevarse directamente a un `DataFrame`:

In [33]:
pipeline = [
    {"$unwind": "$materias"},
    {"$group": {"_id": "$materias.codigo", "promedio": {"$avg": "$materias.nota"}}},
]
df = pd.DataFrame(db.alumnos.aggregate(pipeline))
df.sort_values("promedio", ascending=False)

,_id,promedio
1,SEM,8.800000
0,BD2,8.333333
3,RED,7.000000
2,PRG,6.500000


## 11. Práctica

Ejercicios del apunte y de la diapositiva 18. Las celdas vacías son para resolverlos.

1. Crear la base `instituto` y la colección `alumnos` con al menos 10 documentos, algunos con `materias` y otros sin ellas. *(resuelto en la sección 5.1)*
2. Listar los alumnos de una carrera mostrando solo nombre y legajo, ordenados por apellido.
3. Buscar alumnos con alguna nota mayor o igual a 9.
4. Agregar una materia a un alumno con `$push` y subir 1 punto las notas de una materia con `updateMany`.
5. Calcular el promedio por materia con `aggregate` y comparar con la consulta T-SQL equivalente.
6. Crear un índice único por legajo y comparar `explain()` antes y después (requiere servidor MongoDB real).
7. Para cada uno de los cuatro modelos NoSQL, proponer un caso de uso distinto de los del apunte y justificar.
8. Un sistema de reservas de hoteles con sucursales en varios países, ¿debería priorizar CP o AP? Justificar con el teorema CAP.

## Para llevarse

- NoSQL no reemplaza a SQL: se elige el modelo según el problema.
- **Persistencia políglota:** cada parte del sistema usa el motor que mejor le sirve.
- En MongoDB se modela para la consulta: embeber lo que se lee junto, referenciar lo que se actualiza por separado.

**Próxima clase:** bases de datos vectoriales en SQL Server 2025.

*Bibliografía: Meier y Kaufmann (2019), caps. 1.3, 1.4.3, 4.3 y 7; Sadalage y Fowler (2013); Sarasa (2016), caps. II a VIII.*